In [2]:
import pandas as pd

In [3]:
security_master = pd.read_csv("nse_security_master.csv")
explorer_data = pd.read_csv("nse_explorer_20symbols.csv")

In [4]:
reconciliation = explorer_data.merge(
    security_master,
    on="nse_symbol",
    how="left",
    suffixes=("_explorer", "_master"),
    indicator=True)

print(reconciliation[["nse_symbol", "_merge"]])

    nse_symbol _merge
0   ASIANPAINT   both
1     AXISBANK   both
2   BAJFINANCE   both
3   BHARTIARTL   both
4      HCLTECH   both
5     HDFCBANK   both
6   HINDUNILVR   both
7    ICICIBANK   both
8         INFY   both
9    KOTAKBANK   both
10          LT   both
11      MARUTI   both
12        NTPC   both
13        ONGC   both
14    RELIANCE   both
15        SBIN   both
16   SUNPHARMA   both
17         TCS   both
18       TITAN   both
19       WIPRO   both


In [5]:
#Reconciliation_Summary

In [6]:
total_records = len(reconciliation)
matched_records = (reconciliation["_merge"] == "both").sum()
unmatched_records = (reconciliation["_merge"] == "left_only").sum()
match_pct = round(matched_records / total_records * 100, 2) if total_records > 0 else 0

print("RECONCILIATION_SUMMARY")
print("----------------------")
print("Total Explorer Records:", total_records)
print("Total Matched Records:", matched_records)
print("Total Unmatched Records:", unmatched_records)
print("Total Match Pct:", match_pct)

RECONCILIATION_SUMMARY
----------------------
Total Explorer Records: 20
Total Matched Records: 20
Total Unmatched Records: 0
Total Match Pct: 100.0


In [7]:
#Compare the four fields

In [8]:
print("ISIN mismatches:", (reconciliation["isin_explorer"] != reconciliation["isin_master"]).sum())
print("Company Name mismatches:", (reconciliation["company_name_explorer"] != reconciliation["company_name_master"]).sum())
print("Listing Date mismatches:", (reconciliation["listing_date_explorer"] != reconciliation["listing_date_master"]).sum())
print("Active Status mismatches:", (reconciliation["active_flag_explorer"] != reconciliation["active_flag_master"]).sum())

ISIN mismatches: 0
Company Name mismatches: 0
Listing Date mismatches: 0
Active Status mismatches: 0


In [9]:
validation_report = pd.DataFrame(
    {"Metric":["Total Explorer Records", "Matched Symbols", "Unmatched Symbols", "Symbol Match Rate (%)", "ISIN Mismatches", "Company Name Mismatches", "Listing Date Mismatches", "Active Status Mismatches"],
     "Result":[total_records, matched_records, unmatched_records, match_pct, (reconciliation["isin_explorer"] != reconciliation["isin_master"]).sum(), 
               (reconciliation["company_name_explorer"] != reconciliation["company_name_master"]).sum(),
               (reconciliation["listing_date_explorer"] != reconciliation["listing_date_master"]).sum(),
               (reconciliation["active_flag_explorer"] != reconciliation["active_flag_master"]).sum()]})

In [10]:
validation_report

,Metric,Result
0,Total Explorer Records,20.0
1,Matched Symbols,20.0
2,Unmatched Symbols,0.0
3,Symbol Match Rate (%),100.0
4,ISIN Mismatches,0.0
5,Company Name Mismatches,0.0
6,Listing Date Mismatches,0.0
7,Active Status Mismatches,0.0


In [11]:
# Null-safe field comparisons

field_pairs = {
    "ISIN": ("isin_explorer", "isin_master"),
    "Company Name": ("company_name_explorer", "company_name_master"),
    "Listing Date": ("listing_date_explorer", "listing_date_master"),
    "Active Status": ("active_flag_explorer", "active_flag_master")
}

for field, (explorer_col, master_col) in field_pairs.items():
    mismatches = (
        reconciliation[explorer_col]
        .ne(reconciliation[master_col])
        .fillna(True)
    )
    print(f"{field} mismatches: {mismatches.sum()}")

ISIN mismatches: 0
Company Name mismatches: 0
Listing Date mismatches: 0
Active Status mismatches: 0
